# Normalize GLASS Pathogen-Antibiotic Pairs

This notebook reads `archive/legacy_assets/GLASS list.xlsx`, normalizes GLASS species and antibiotic labels to the project naming system, and writes a clean CSV used by `8_antibiotic_class_scientific_interest.ipynb`.



In [1]:
%config InlineBackend.figure_format = 'retina'

import re
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "amr").exists():
        PROJECT_ROOT = candidate
        break
    nested_candidate = candidate / "Antimicrobial Resistance"
    if (nested_candidate / "amr").exists():
        PROJECT_ROOT = nested_candidate
        break

if PROJECT_ROOT is None:
    raise ModuleNotFoundError("Could not locate the project root containing the 'amr' package.")


In [2]:
ARCHIVE_DIR = PROJECT_ROOT / "archive"
LEGACY_ASSETS_DIR = ARCHIVE_DIR / "legacy_assets"

ANTIBIOTIC_METADATA_PATH = LEGACY_ASSETS_DIR / "antibiotic_list2.xlsx"
YEARLY_PATH = LEGACY_ASSETS_DIR / "pmids_results_positive_yearly_refined.parquet"
GLASS_LIST_PATH = LEGACY_ASSETS_DIR / "GLASS list.xlsx"

ANTIBIOTIC_GROUP_COLUMN = "Class"
GLASS_NORMALIZED_PATH = LEGACY_ASSETS_DIR / f"GLASS_pairs_normalized_{ANTIBIOTIC_GROUP_COLUMN.lower().replace(' ', '_')}.csv"


In [3]:
antibiotics_df = pd.read_excel(ANTIBIOTIC_METADATA_PATH)
if ANTIBIOTIC_GROUP_COLUMN == "Antibiotic":
    mapping_df = antibiotics_df[["Antibiotic"]].copy()
    mapping_df["antibiotic_group"] = mapping_df["Antibiotic"]
else:
    mapping_df = antibiotics_df[["Antibiotic", ANTIBIOTIC_GROUP_COLUMN]].copy()
    mapping_df = mapping_df.rename(columns={ANTIBIOTIC_GROUP_COLUMN: "antibiotic_group"})

mapping_df["Antibiotic"] = mapping_df["Antibiotic"].astype(str).str.strip()
mapping_df["antibiotic_group"] = mapping_df["antibiotic_group"].astype(str).str.strip()
mapping_df = mapping_df.replace({"": pd.NA}).dropna(subset=["Antibiotic", "antibiotic_group"])
mapping_df = mapping_df.drop_duplicates(subset=["Antibiotic"]).reset_index(drop=True)

yearly_df_raw = pd.read_parquet(YEARLY_PATH, columns=["pathogen"])
available_pathogens = sorted(yearly_df_raw["pathogen"].dropna().astype(str).str.strip().unique())

print(f"Available yearly pathogens: {len(available_pathogens):,}")
print(f"Mapped antibiotics: {len(mapping_df):,}")
print(f"Mapped {ANTIBIOTIC_GROUP_COLUMN.lower()} values: {mapping_df['antibiotic_group'].nunique():,}")


Available yearly pathogens: 577
Mapped antibiotics: 245
Mapped class values: 38


In [4]:
glass_raw_df = pd.read_excel(GLASS_LIST_PATH, sheet_name="raw")
glass_raw_df = glass_raw_df.rename(
    columns={
        "species": "glass_species",
        "antibiotics": "glass_antibiotic",
        "antibiotic group": "glass_antibiotic_group",
        "to check": "glass_to_check",
    }
)

for col in ["glass_species", "glass_antibiotic", "glass_antibiotic_group", "glass_to_check"]:
    glass_raw_df[col] = glass_raw_df[col].astype("string").str.strip()

glass_source_df = glass_raw_df.dropna(subset=["glass_species", "glass_antibiotic"]).drop_duplicates(
    subset=["glass_species", "glass_antibiotic", "glass_antibiotic_group", "glass_to_check"]
)

display(glass_source_df.head(20))


,specimen,glass_species,glass_antibiotic,glass_antibiotic_group,glass_to_check
0,blood,Acinetobacter spp.,Tigecycline,tetracyclines,<NA>
1,blood,Acinetobacter spp.,Gentamicin,aminoglycosides,<NA>
2,blood,Acinetobacter spp.,Imipenem,carbapenems,<NA>
3,blood,Acinetobacter spp.,Colistin,polymyxins,<NA>
4,blood,Acinetobacter spp.,minocycline,<NA>,<NA>
5,blood,Acinetobacter spp.,amikacin,<NA>,<NA>
6,blood,Acinetobacter spp.,meropenem,<NA>,<NA>
7,blood,Acinetobacter spp.,doripenem,<NA>,<NA>
8,blood,E. coli,Co-trimoxazole,sufonamides,and trimethoprim?
9,blood,E. coli,Ciprofloxacin,fluoroquinolones,<NA>


In [5]:
def _norm_key(value):
    if pd.isna(value):
        return pd.NA
    return re.sub(r"[^a-z0-9]+", "", str(value).casefold())


def _normalize_name(value):
    return re.sub(r"\s+", " ", str(value).strip())


pathogen_lookup = {_norm_key(pathogen): pathogen for pathogen in available_pathogens}
pathogen_aliases = {
    "ecoli": "Escherichia coli",
}


def _expand_glass_species(glass_species):
    species = _normalize_name(glass_species).rstrip(".")
    species_key = _norm_key(species)
    if species_key in pathogen_aliases:
        return [pathogen_aliases[species_key]]
    if species_key in pathogen_lookup:
        return [pathogen_lookup[species_key]]
    if species.casefold().endswith(" spp"):
        genus = species[:-4].strip()
        matches = [pathogen for pathogen in available_pathogens if pathogen.casefold().startswith(genus.casefold() + " ")]
        if matches:
            return matches
    return []


meta_cols = ["Antibiotic", "Class", "Simplified categories", "High level categories"]
glass_meta_df = antibiotics_df[meta_cols].copy()
for col in meta_cols:
    glass_meta_df[col] = glass_meta_df[col].astype("string").str.strip()

drug_lookup_df = glass_meta_df.dropna(subset=["Antibiotic"]).copy()
drug_lookup_df["drug_key"] = drug_lookup_df["Antibiotic"].map(_norm_key)
drug_lookup_df = drug_lookup_df.drop_duplicates(subset=["drug_key"], keep="first")
drug_lookup = drug_lookup_df.set_index("drug_key").to_dict("index")

class_lookup = {
    _norm_key(group): group
    for group in mapping_df["antibiotic_group"].dropna().astype(str).map(_normalize_name).unique()
}
antibiotic_aliases = {
    "amoxicillinclavulanicacid": "Amoxicillin/clavulanic-acid",
    "cotrimoxazole": "Sulfamethoxazole/trimethoprim",
    "imipenem": "Imipenem/cilastatin",
    "oxacilline": "Oxacillin",
    "penicilling": "Benzylpenicillin",
}
class_aliases = {
    "aminocyclitol": "Aminocyclitols",
    "sufonamides": "Sulfonamides",
    "nitrofuranderivatives": "Nitrofuran-derivatives",
    "betalactamasesensitivepenicillins": "Penicillins",
    "betalactamaseresistantpenicillins": "Penicillins",
    "penicillinswithextendedspectrum": "Penicillins",
    "combinationsofpenicillinsincludingbetalactamaseinhibitors": "Beta-lactam/beta-lactamase-inhibitor",
    "combinationsofpenicillinsincludingbetalactamaseinhibitorssulfonamidesandtrimethoprim": "Sulfonamide-trimethoprim-combinations",
    "andtrimethoprim": "Sulfonamide-trimethoprim-combinations",
}


def _metadata_row_for_drug(drug_name):
    drug_key = _norm_key(drug_name)
    if drug_key in drug_lookup:
        return drug_lookup[drug_key], "Antibiotic"
    alias = antibiotic_aliases.get(drug_key)
    if alias is not None:
        alias_key = _norm_key(alias)
        if alias_key in drug_lookup:
            return drug_lookup[alias_key], "Antibiotic alias"
    return None, None


def _resolve_antibiotic(row):
    meta_row, matched_on = _metadata_row_for_drug(row.glass_antibiotic)
    if meta_row is not None:
        antibiotic_name = meta_row["Antibiotic"]
        antibiotic_group = meta_row["Class"] if ANTIBIOTIC_GROUP_COLUMN == "Class" else meta_row.get(ANTIBIOTIC_GROUP_COLUMN)
    else:
        antibiotic_name = pd.NA
        antibiotic_group = pd.NA
        matched_on = pd.NA
        for source_col, source_label in [
            ("glass_antibiotic_group", "GLASS antibiotic group"),
            ("glass_to_check", "GLASS to check"),
        ]:
            value = getattr(row, source_col)
            value_key = _norm_key(value)
            if pd.isna(value_key):
                continue
            if value_key in class_aliases:
                antibiotic_group = class_aliases[value_key]
                matched_on = source_label
                break
            if value_key in class_lookup:
                antibiotic_group = class_lookup[value_key]
                matched_on = source_label
                break

    if ANTIBIOTIC_GROUP_COLUMN == "Antibiotic":
        antibiotic = antibiotic_name
    else:
        antibiotic = antibiotic_group
    return antibiotic, antibiotic_name, antibiotic_group, matched_on


In [6]:
glass_rows = []
unresolved_rows = []
for row in glass_source_df.itertuples(index=False):
    pathogens = _expand_glass_species(row.glass_species)
    antibiotic, antibiotic_name, antibiotic_group, matched_on = _resolve_antibiotic(row)
    if not pathogens or pd.isna(antibiotic):
        unresolved_rows.append(
            {
                "glass_species": row.glass_species,
                "glass_antibiotic": row.glass_antibiotic,
                "glass_antibiotic_group": row.glass_antibiotic_group,
                "glass_to_check": row.glass_to_check,
                "pathogen_resolved": bool(pathogens),
                "antibiotic_resolved": not pd.isna(antibiotic),
            }
        )
        continue
    for pathogen in pathogens:
        glass_rows.append(
            {
                "pathogen": pathogen,
                "antibiotic": antibiotic,
                "glass_species": row.glass_species,
                "glass_antibiotic": row.glass_antibiotic,
                "glass_antibiotic_name": antibiotic_name,
                "glass_antibiotic_group": antibiotic_group,
                "matched_on": matched_on,
            }
        )

GLASS_pairs_df = pd.DataFrame(glass_rows).drop_duplicates(subset=["pathogen", "antibiotic"]).reset_index(drop=True)
unresolved_GLASS_df = pd.DataFrame(unresolved_rows).drop_duplicates().reset_index(drop=True)

print(f"Loaded {len(glass_source_df):,} GLASS rows from {GLASS_LIST_PATH}.")
print(f"Resolved {len(GLASS_pairs_df):,} pathogen-{ANTIBIOTIC_GROUP_COLUMN.lower()} pairs.")
if not unresolved_GLASS_df.empty:
    print(f"Unresolved GLASS rows: {len(unresolved_GLASS_df):,}")
    display(unresolved_GLASS_df)

display(GLASS_pairs_df)


Loaded 142 GLASS rows from /Users/fabriziocosta/Resilio Sync/Sync/Projects/ACTIVE/Pathogen Atlas/Antimicrobial Resistance/archive/legacy_assets/GLASS list.xlsx.
Resolved 128 pathogen-class pairs.


,pathogen,antibiotic,glass_species,glass_antibiotic,glass_antibiotic_name,glass_antibiotic_group,matched_on
0,Acinetobacter baumannii,Glycylcyclines,Acinetobacter spp.,Tigecycline,Tigecycline,Glycylcyclines,Antibiotic
1,Acinetobacter bereziniae,Glycylcyclines,Acinetobacter spp.,Tigecycline,Tigecycline,Glycylcyclines,Antibiotic
2,Acinetobacter calcoaceticus,Glycylcyclines,Acinetobacter spp.,Tigecycline,Tigecycline,Glycylcyclines,Antibiotic
3,Acinetobacter haemolyticus,Glycylcyclines,Acinetobacter spp.,Tigecycline,Tigecycline,Glycylcyclines,Antibiotic
4,Acinetobacter johnsonii,Glycylcyclines,Acinetobacter spp.,Tigecycline,Tigecycline,Glycylcyclines,Antibiotic
...,...,...,...,...,...,...,...
123,Neisseria gonorrhoeae,Third-generation-cephalosporins,Neisseria gonorrhoeae,Ceftriaxone,Ceftriaxone,Third-generation-cephalosporins,Antibiotic
124,Neisseria gonorrhoeae,Macrolides,Neisseria gonorrhoeae,Azithromycin,Azithromycin,Macrolides,Antibiotic
125,Neisseria gonorrhoeae,Aminocyclitols,Neisseria gonorrhoeae,spectinomycin,Spectinomycin,Aminocyclitols,Antibiotic
126,Neisseria gonorrhoeae,Fluoroquinolones,Neisseria gonorrhoeae,Ciprofloxacin,Ciprofloxacin,Fluoroquinolones,Antibiotic


In [7]:
GLASS_pairs_df.to_csv(GLASS_NORMALIZED_PATH, index=False)
print(f"Saved normalized GLASS pairs to {GLASS_NORMALIZED_PATH}")


Saved normalized GLASS pairs to /Users/fabriziocosta/Resilio Sync/Sync/Projects/ACTIVE/Pathogen Atlas/Antimicrobial Resistance/archive/legacy_assets/GLASS_pairs_normalized_class.csv
